# PANNs submission within a fixed twelve-hour deadline
Reuse the existing A100 and verified cache. R1/R3 use seed 41, max 200 epochs, patience 15. Full selection evaluation, an offline submission ZIP, and real entry-point QA follow. No official submission.

In [ ]:
import hashlib, json, os, subprocess, time
from pathlib import Path
config = json.loads(Path(os.environ['DACON_EXPERIMENT_CONFIG']).read_text())
assert config['workspace'] == 'wsp-nvxqw1jf9zdv'
assert config['deadline_epoch'] == config['started_epoch'] + 12 * 3600
assert config['started_epoch'] <= time.time() < config['deadline_epoch']


In [ ]:
for filename, expected in config['inputs'].items():
    with Path(filename).open('rb') as stream:
        assert hashlib.file_digest(stream, 'sha256').hexdigest() == expected
controller = json.loads(os.environ['PANNS_CONTROLLER_CONTRACT'])
assert controller['workspace'] == config['workspace']
assert controller['deadline_epoch'] == config['deadline_epoch']
assert controller['bundle_sha256'] == config['bundle_sha256']


In [ ]:
env = dict(os.environ, PANNS_DEADLINE_EPOCH=str(config['deadline_epoch']),
           PANNS_DEADLINE_STATE=config['deadline_state'], PANNS_PAUSE_ON_EXIT='1')
with Path(config['log']).open('a') as log:
    subprocess.run(['python3', config['guard'], config['launcher']], env=env,
                   stdout=log, stderr=subprocess.STDOUT, check=True)


In [ ]:
result = json.loads(Path(config['completion']).read_text())
assert result['deadline_epoch'] == config['deadline_epoch']
assert result['finished_epoch'] <= config['deadline_epoch']
print(json.dumps(result))
